# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zainab-Aijaz/WEEK1_ML_Assignment_FlyRank_Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [3]:
from huggingface_hub import login

login()

In [4]:
from google.colab import userdata
import os

os.environ["HF-TOKEN"] = userdata.get("HF-TOKEN")

In [5]:
from datasets import load_dataset
import duckdb


fact_content_ds = load_dataset("FlyRank/internship-warehouse", "fact_content_daily_performance", split="train")
fact_content = fact_content_ds.data.table   # Arrow, not pandas — much lighter

dim_content_ds = load_dataset("FlyRank/internship-warehouse", "dim_content", split="train")
dim_content = dim_content_ds.data.table

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 3.29MB            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 4.41MB            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 3.22MB            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

...

Generating train split:   0%|          | 0/78835655 [00:00<?, ? examples/s]

Loading dataset shards:   0%|          | 0/39 [00:00<?, ?it/s]

dim_content.parquet: reconstructing file:   0%|          |  0.00B / 19.6MB            

dim_content.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/519606 [00:00<?, ? examples/s]

In [4]:
from google.colab import userdata
import os

token = userdata.get("HF-TOKEN")
print("Token loaded:", token is not None, "| length:", len(token) if token else 0)

Token loaded: True | length: 37


from google.colab import userdata
import duckdb

token = userdata.get("HF-TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE SECRET (
    TYPE huggingface,
    TOKEN '{token}'
)
""") # accept the gate in-browser first, then paste a READ token
rel = "hf://datasets/FlyRank/internship-warehouse"
con.sql(f"SELECT COUNT(*) FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')")

In [5]:
import duckdb
# grain check on dim_content — is it really one row per content+client?
duckdb.sql("""
    SELECT client_hash_id, content_hash_id, COUNT(*) c
    FROM dim_content
    GROUP BY client_hash_id, content_hash_id
    HAVING c > 1
    LIMIT 5
""")
# if rows come back, a content_hash_id maps to >1 keyword/url — the real grain is
# content+client+keyword, not content+client. Check before assuming.

┌────────────────┬─────────────────┬───────┐
│ client_hash_id │ content_hash_id │   c   │
│    varchar     │     varchar     │ int64 │
├────────────────┴─────────────────┴───────┤
│                  0 rows                  │
└──────────────────────────────────────────┘

*This is a nice contrast to the fact table, by the way — dim_content's grain held` (0 duplicates)`, but fact_content's grain didn't` (you found c = 2 duplicates there)`. So the honest line for your contract is: dim_content is clean, fact_content has duplicate rows that need deduplication before use. Worth stating both outcomes side by side — it shows you actually checked each table rather than assuming they'd behave the same way.*

In [6]:
# grain check on fact table
duckdb.sql("""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) c
    FROM fact_content
    GROUP BY client_hash_id, content_hash_id, report_date
    HAVING c > 1
    LIMIT 5
""")
# expect 0 rows back

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────────────┬──────────────────────────┬─────────────┬───────┐
│     client_hash_id      │     content_hash_id      │ report_date │   c   │
│         varchar         │         varchar          │    date     │ int64 │
├─────────────────────────┼──────────────────────────┼─────────────┼───────┤
│ client_1a730cb2640a1abf │ content_6604767cde89152e │ 2026-06-13  │     2 │
│ client_1a730cb2640a1abf │ content_b5aec9a8a2ee7fb0 │ 2026-06-13  │     2 │
│ client_1a8bf67cad4ee525 │ content_2630830d5f397c6c │ 2026-06-15  │     2 │
│ client_8ddc46da5414ffd8 │ content_a9789f58505f4f9b │ 2026-06-13  │     2 │
│ client_8ddc46da5414ffd8 │ content_d07f96c572dc6f2a │ 2026-06-13  │     2 │
└─────────────────────────┴──────────────────────────┴─────────────┴───────┘

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## **Feature (knowable before the decision moment, from dim_content):**

 *content_type, search_volume, competition, competition_level, cpc, main_intent, backlinks, category_count, char_count, word_count, content_created_date, content_updated_date, last_optimized_date, optimization_eligible_date, is_published.*


##**Feature, rolled up from fact_content_daily_performance**

*— but only using days strictly BEFORE the decision date: prior gsc_clicks, gsc_impressions, gsc_avg_position, ga4_sessions, ga4_engaged_sessions trends.*

##**Label / proxy:**

*whether gsc_clicks decline over the 30 days after the decision date — computed only from days after, never before.*

##**Context**

 (for joining/grouping/filtering, never fed to the model): client_hash_id, content_hash_id, keyword_hash_id, url_hash_id, report_date.

## **Excluded, with why:**

* provider_used, model_used — describes who/what authored the content internally, not a real signal about page performance.


* is_deleted — a lifecycle/product-decision flag, not a performance signal; deleted pages shouldn't be in the scoring pool at all.


* ai_chatgpt, ai_perplexity, ai_gemini, ai_copilot, ai_claude, ai_meta, ai_other — too granular per-AI-source breakdown for this lane; sessions_ai at the aggregate level is enough, and using the same-window breakdown risks overlapping with the label window.


* client_has_gsc, client_has_ga4, gsc_data_available, ga4_data_available — these are availability flags, used to filter rows, not fed to the model as predictive features.





## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [8]:
# counts + date span for the month
display(duckdb.sql("""
    SELECT COUNT(*) AS total_rows, MIN(report_date) AS first_date, MAX(report_date) AS last_date
    FROM fact_content
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
"""))

# availability: how many rows actually have usable GSC data
display(duckdb.sql("""
    SELECT COUNT(*) AS available_rows
    FROM fact_content
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
      AND gsc_data_available IS TRUE
"""))

# missingness overall
display(duckdb.sql("""
    SELECT
      AVG(CASE WHEN gsc_avg_position IS NULL THEN 1.0 ELSE 0 END) AS pct_missing_position,
      AVG(CASE WHEN ga4_sessions IS NULL THEN 1.0 ELSE 0 END) AS pct_missing_ga4_sessions
    FROM fact_content
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
"""))

# missingness by content_type (patterned gaps check)
display(duckdb.sql("""
    SELECT d.content_type,
           AVG(CASE WHEN f.gsc_avg_position IS NULL THEN 1.0 ELSE 0 END) AS pct_missing_position
    FROM fact_content f
    JOIN dim_content d USING (client_hash_id, content_hash_id)
    WHERE f.report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
    GROUP BY d.content_type
"""))

# windows per client — histories rarely start together
display(duckdb.sql("""
    SELECT client_hash_id, MIN(report_date) AS first_date, MAX(report_date) AS last_date
    FROM fact_content
    GROUP BY client_hash_id
    LIMIT 10
"""))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────┬────────────┐
│ total_rows │ first_date │ last_date  │
│   int64    │    date    │    date    │
├────────────┼────────────┼────────────┤
│    9841378 │ 2026-03-01 │ 2026-03-31 │
└────────────┴────────────┴────────────┘

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────┐
│ available_rows │
│     int64      │
├────────────────┤
│        3611061 │
└────────────────┘

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────────────┬──────────────────────────┐
│ pct_missing_position │ pct_missing_ga4_sessions │
│        double        │          double          │
├──────────────────────┼──────────────────────────┤
│   0.6330736407035681 │      0.30673966592889734 │
└──────────────────────┴──────────────────────────┘

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────────┬──────────────────────┐
│    content_type    │ pct_missing_position │
│      varchar       │        double        │
├────────────────────┼──────────────────────┤
│ keyword article    │   0.5789910455652295 │
│ feedly article     │   0.9467437533227007 │
│ comparison article │   0.3778323155071058 │
└────────────────────┴──────────────────────┘

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────────────┬────────────┬────────────┐
│     client_hash_id      │ first_date │ last_date  │
│         varchar         │    date    │    date    │
├─────────────────────────┼────────────┼────────────┤
│ client_9958f0a7ae1df715 │ 2025-01-27 │ 2026-06-30 │
│ client_73cda7b4e4f265ea │ 2025-02-11 │ 2026-06-30 │
│ client_fef1a8f436438636 │ 2025-03-11 │ 2026-06-30 │
│ client_c182d11e4862a37d │ 2025-06-21 │ 2026-06-30 │
│ client_62f4a7e64f5e0096 │ 2025-06-07 │ 2026-06-30 │
│ client_a2eeb8899886adde │ 2025-07-06 │ 2026-06-30 │
│ client_d211cb07b9059bab │ 2025-07-07 │ 2026-06-30 │
│ client_8ae2bfb5aa1ffa1e │ 2025-07-28 │ 2026-06-30 │
│ client_08a6a72ff48e62c0 │ 2025-09-24 │ 2026-06-30 │
│ client_4a18d1793d92fb84 │ 2025-09-24 │ 2026-06-30 │
├─────────────────────────┴────────────┴────────────┤
│ 10 rows                                 3 columns │
└───────────────────────────────────────────────────┘

# ***Features***

In [8]:
features = duckdb.sql("""
    WITH prior_position AS (
        SELECT client_hash_id, content_hash_id,
               AVG(gsc_avg_position) AS gsc_avg_position_prior
        FROM fact_content
        WHERE report_date BETWEEN DATE '2026-02-01' AND DATE '2026-02-28'
          AND gsc_data_available IS TRUE
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT
        d.client_hash_id,
        d.content_hash_id,
        d.word_count,
        d.search_volume,
        d.competition_level,
        DATE_DIFF('day', d.content_created_date, DATE '2026-03-01') AS content_age_days,
        p.gsc_avg_position_prior
    FROM dim_content d
    LEFT JOIN prior_position p
        USING (client_hash_id, content_hash_id)
    WHERE d.is_published IS TRUE
      AND d.is_deleted IS FALSE
""")


In [10]:
features_df = features.df()
features_df

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,word_count,search_volume,competition_level,content_age_days,gsc_avg_position_prior
0,client_04660893ae39614a,content_c81a0401fc0e870a,2636,30,HIGH,-101,NaN
1,client_04660893ae39614a,content_c8219549d6762429,2410,10,MEDIUM,-78,NaN
2,client_04660893ae39614a,content_c8a199ce47857f68,2881,2400,HIGH,-76,NaN
3,client_04660893ae39614a,content_c8eb16f7b396c9b0,2631,0,LOW,-127,NaN
4,client_04660893ae39614a,content_c8f4a0373eaafa45,2934,10,HIGH,-101,NaN
...,...,...,...,...,...,...,...
411535,client_b77d0d5f08f05e64,content_021c879a22bb0470,2608,0,LOW,-54,NaN
411536,client_b77d0d5f08f05e64,content_022f85b0b3fd45ec,2226,0,LOW,-124,NaN
411537,client_b77d0d5f08f05e64,content_0231d863fae5c35d,2580,10,LOW,-85,NaN
411538,client_b77d0d5f08f05e64,content_02329de5cfd67f78,2704,0,LOW,-115,NaN


In [11]:
duckdb.sql("""
    SELECT MIN(content_created_date), MAX(content_created_date), COUNT(*)
    FROM dim_content
    WHERE content_created_date > DATE '2026-03-01'
""")

┌───────────────────────────┬───────────────────────────┬──────────────┐
│ min(content_created_date) │ max(content_created_date) │ count_star() │
│           date            │           date            │    int64     │
├───────────────────────────┼───────────────────────────┼──────────────┤
│ 2026-03-02                │ 2026-07-06                │       113174 │
└───────────────────────────┴───────────────────────────┴──────────────┘

In [12]:
duckdb.sql("""
    SELECT
        COUNT(*) AS total,
        SUM(CASE WHEN content_created_date > DATE '2026-03-01' THEN 1 ELSE 0 END) AS created_after_march
    FROM dim_content
""")

┌────────┬─────────────────────┐
│ total  │ created_after_march │
│ int64  │       int128        │
├────────┼─────────────────────┤
│ 519606 │              113174 │
└────────┴─────────────────────┘

In [13]:
features = duckdb.sql("""
    WITH prior_position AS (
        SELECT client_hash_id, content_hash_id,
               AVG(gsc_avg_position) AS gsc_avg_position_prior
        FROM fact_content
        WHERE report_date BETWEEN DATE '2026-02-01' AND DATE '2026-02-28'
          AND gsc_data_available IS TRUE
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT
        d.client_hash_id,
        d.content_hash_id,
        d.word_count,
        d.search_volume,
        d.competition_level,
        DATE_DIFF('day', d.content_created_date, DATE '2026-03-01') AS content_age_days,
        p.gsc_avg_position_prior
    FROM dim_content d
    LEFT JOIN prior_position p
        USING (client_hash_id, content_hash_id)
    WHERE d.is_published IS TRUE
      AND d.is_deleted IS FALSE
      AND d.content_created_date <= DATE '2026-03-01'
""")

features.df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,word_count,search_volume,competition_level,content_age_days,gsc_avg_position_prior
0,client_08a6a72ff48e62c0,content_0726359e123f01e1,<NA>,90,LOW,241,1.500000
1,client_08a6a72ff48e62c0,content_072bcc8238b30d62,2395,0,LOW,214,11.046023
2,client_08a6a72ff48e62c0,content_072fad2a83b01e63,3073,10,LOW,214,7.396809
3,client_08a6a72ff48e62c0,content_073bc74a54098a90,<NA>,10,LOW,230,56.228571
4,client_08a6a72ff48e62c0,content_073ed09621daa9bb,<NA>,1900,HIGH,313,3.694444
...,...,...,...,...,...,...,...
303327,client_7eafe750768f0ac2,content_4f939132eb4ac985,2814,10,LOW,4,NaN
303328,client_7eafe750768f0ac2,content_9e0c10cbc337953d,2844,10,LOW,4,NaN
303329,client_7eafe750768f0ac2,content_b71d3beb4c40015e,2413,0,LOW,4,NaN
303330,client_7eafe750768f0ac2,content_b72ad87595491ef8,2664,10,LOW,4,NaN


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

# **Data limits**

This data can never tell me whether a refresh caused recovery — no page here was refreshed and observed afterward, so "declining → needs refresh" stays a proxy, not proven cause and effect.
client_has_gsc / client_has_ga4 show that not every client has both data sources — a client without GA4 will have ga4_sessions, ga4_engaged_sessions, etc. as structurally missing, not zero engagement. Any feature built from GA4 needs to check this flag first, or it'll misread "no tracking" as "no engagement."
content_created_date vs keyword_created_date may not align — a page could be older than the keyword record or vice versa, so content age needs to be computed from the right date, not assumed.
Clients don't share a common history start — comparing "declining in March" across clients isn't apples-to-apples unless each client's own data start date is checked first (see the windows-per-client query above)

*GSC data availability is not random — it's concentrated by content_type. "Feedly article" pages are missing gsc_avg_position in ~95% of rows, versus ~38% for "comparison article" pages. Any model trained without accounting for this will underrepresent feedly articles, not because they're less at-risk, but because there's far less signal to learn from for that content type.*

*~22% of pages in dim_content (113,174 of 519,606) were created after the March 2026 decision window and were excluded from the March feature frame — these pages simply didn't exist yet at decision time, so no "as-of" feature can honestly be computed for them.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.